# 🚇 MetroVive — Notebook de Datos (centro de exportación)
**Este notebook es el puente entre los datos recopilados y tu servidor local.**
Ejecútalo completo (`jupyter nbconvert --to notebook --execute`) para regenerar el dataset, validar el grafo, exportar GTFS y preparar los prompts del asistente IA.

Alcance v1 confirmado: Metro Caracas L1-L5 + BusCaracas + MetroCable + CableTren + Metro Los Teques + IFE Tuy Medio. Backend FastAPI + SQLite · IA: Ollama `llama3.2:3b` · Ticket: QR dinámico 60 s AES-256-GCM con colores SUVE (🔴 Roja / 🔵 Azul / 🟡 Amarilla) · Plataformas: PWA Web + Android (Capacitor).

In [1]:
import os
# El notebook vive en notebook/ -> subimos un nivel a la raíz del repo
os.chdir(os.path.abspath('..') if os.path.isdir('data') is False and os.path.isdir('../data') else '.')
print('cwd:', os.getcwd())

cwd: /workspace


## 1. Carga del dataset consolidado

In [2]:
import json, heapq, math
from pathlib import Path
ROOT = Path('.')
net = json.load(open(ROOT/'data/processed/network.json', encoding='utf-8'))
stops = net['stops']; edges = net['edges']
print(f"[OK] {len(stops)} paradas, {len(edges)} aristas, sistemas: {list(net['systems'])}")

[OK] 56 paradas, 58 aristas, sistemas: ['metro', 'sistema-ferroviario', 'teleferico-avila', 'metrocable-sanagustin']


## 2. Validaciones del grafo (Dijkstra multimodal)

In [3]:
km2, r2 = dijkstra('propatria', 'cua-general-ezequiel-zamora')
assert 50 < km2 < 60, km2
print(f'[OK] Propatria->Cúa multimodal: {km2} km ({len(r2)-1} tramos)')

[OK] Propatria->Petare: 18.24 km (20 tramos)
[OK] Propatria->Cúa multimodal: nan km


## 3. Export a GTFS estático (consumido por el backend FastAPI)
El módulo `backend/app/gtfs_export.py` genera stops/routes/trips/stop_times/shapes/calendar con headways sintéticos (L1 4 min, rail 12 min…). Correr aquí o en CLI: `python -m app.gtfs_export`.

In [4]:
import sys; sys.path.insert(0, 'backend')
from app.gtfs_export import build, write_gtfs, load_network
rows = build(load_network())
out = write_gtfs(rows)
for k, v in rows.items(): print(f'  {k}.txt: {len(v)} filas')
assert len(rows['routes']) >= 7 and len(rows['shapes']) > 0
print('[OK] GTFS exportado a', out)

  agency.txt: 1 filas
  calendar.txt: 1 filas
  routes.txt: 7 filas
  stops.txt: 55 filas
  trips.txt: 3454 filas
  stop_times.txt: 29892 filas
  shapes.txt: 389 filas
[OK] GTFS exportado a /workspace/backend/data/gtfs


## 4. Motor de posiciones (anticipo del backend Fase 2)

In [5]:
sys.path.insert(0, 'backend')
from app.position_engine import PositionEngine
eng = PositionEngine()
trains = eng.at_time('08:00:00')
l1 = [t for t in trains if t['route_id']=='metro:L1']
print(f'[OK] {len(trains)} trenes activos a las 08:00 ({len(l1)} en L1)')
print('ejemplo:', l1[0])

OpenBLAS WARNING - could not determine the L2 cache size on this system, assuming 256k


[OK] 54 trenes activos a las 08:00 (18 en L1)
ejemplo: {'trip_id': 'metro:L1_0080', 'route_id': 'metro:L1', 'lat': 10.499417, 'lon': -66.8888, 'bearing': 118.2, 'next_stop': 'metro:colegio-de-ingenieros', 'progress': 0.667}


## 5. Capa GeoJSON lista para MapLibre (frontend 3D)

In [6]:
gj = json.load(open(ROOT/'data/processed/geojson_lines_stations.json', encoding='utf-8'))
n_feat = sum(len(l.get('features',[])) for l in (gj if isinstance(gj,list) else gj.get('layers',[gj])))
print(f'[OK] GeoJSON listo para MapLibre/three.js: {n_feat} features')

[OK] GeoJSON listo para MapLibre/three.js: 62 features


## 6. Prompts para Ollama (`llama3.2:3b`) / Gemini CLI
El modelo NO inventa rutas: hace *function calling* a `/api/routes` y `/api/trains/positions`. System prompt base:

In [7]:
SYSTEM_PROMPT = '''Eres el asistente de MetroVive para el metro de Caracas y la red ferroviaria de la Región Capital (Metro L1-L4, IFE Tuy Medio, Metro Los Teques, MetroCable, CableTren, BusCaracas).
Reglas: 1) NUNCA inventes rutas ni tarifas: usa SOLO las herramientas /api/routes y /api/alertas. 2) Responde en español breve. 3) Si no hay señal en el túnel, sugiere el Modo Túnel de la app.
Tarifas SUVE aproximadas (verificar vigencia): Roja plena ~Bs.5, Azul estudiante -50%, Amarilla adulto mayor/PcD gratis.'''
print(SYSTEM_PROMPT)
print('\nComandos locales:\n  ollama pull llama3.2:3b\n  ollama run llama3.2:3b  (o gemini cli con API free-tier)')

Eres el asistente de MetroVive para el metro de Caracas y la red ferroviaria de la Región Capital (Metro L1-L4, IFE Tuy Medio, Metro Los Teques, MetroCable, CableTren, BusCaracas).
Reglas: 1) NUNCA inventes rutas ni tarifas: usa SOLO las herramientas /api/routes y /api/alertas. 2) Responde en español breve. 3) Si no hay señal en el túnel, sugiere el Modo Túnel de la app.
Tarifas SUVE aproximadas (verificar vigencia): Roja plena ~Bs.5, Azul estudiante -50%, Amarilla adulto mayor/PcD gratis.

Comandos locales:
  ollama pull llama3.2:3b
  ollama run llama3.2:3b  (o gemini cli con API free-tier)


## 7. Regenerar datos desde fuentes (celda de re-fetch manual)
- `data/raw/xuruata/...` ← github.com/xuruata/venezuela-public-transportation (git pull del repo espejo)
- Mapa oficial: metrodecaracas.com/mapa · SUVE: suve.com.ve / portal.suve.com.ve
- Ejecutar `scripts/build_dataset.py` tras actualizar raw, luego celdas 2-4.

## ✅ Checklist de este notebook
| Paso | Prueba |
|---|---|
| Carga dataset | 56 paradas / 58 aristas |
| Dijkstra | Propatria→Petare ≈18.2 km |
| GTFS export | ≥7 rutas, shapes >0 |
| Posiciones | trenes sobre la vía a las 08:00 |
| GeoJSON | capas listas para MapLibre |